# ATDL Phase 3 — Pretrained YOLO Baseline (E0)

This trains the standard pretrained YOLOv8n detector on the Phase 2 training split and evaluates on validation only. The test split stays untouched. Training is opt-in because it downloads model weights and runs a GPU workload: install a compatible PyTorch and Ultralytics build, then set RUN_TRAINING = True in the configuration cell.

## 1. Project paths and experiment settings

In [1]:
from pathlib import Path
import json
import hashlib
import shutil
import os
import platform
import subprocess
from datetime import datetime, timezone
import pandas as pd
import yaml

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "requirements-local.txt").is_file() and (p / "configs").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the atdl_final_project folder in VS Code.")
DATA_YAML = PROJECT_ROOT / "configs" / "neu_det_yolo.yaml"
PHASE2_SUMMARY = PROJECT_ROOT / "results" / "phase2" / "phase2_dataset_summary.json"
RUN_DIR = PROJECT_ROOT / "outputs" / "checkpoints" / "phase3_baseline"
METRICS_PATH = PROJECT_ROOT / "results" / "phase3_baseline_metrics.json"
RUN_PATH = RUN_DIR / "E0_baseline"
SIGNATURE_PATH = RUN_PATH / "run_signature.json"

MODEL_WEIGHTS = "yolov8n.pt"
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
SEED = 42
PATIENCE = 12
RUN_TRAINING = True
FORCE_RETRAIN = False

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_files = [DATA_YAML, PROJECT_ROOT / "data" / "splits" / "phase2_image_split.csv",
                PROJECT_ROOT / "results" / "phase2" / "yolo_conversion_manifest.csv"]
dataset_fingerprint = hashlib.sha256("|".join(sha256_file(path) for path in source_files).encode()).hexdigest()
weights_path = PROJECT_ROOT / MODEL_WEIGHTS
training_signature = {
    "dataset_fingerprint": dataset_fingerprint,
    "model": MODEL_WEIGHTS,
    "model_weights_sha256": sha256_file(weights_path) if weights_path.is_file() else None,
    "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH_SIZE,
    "seed": SEED, "patience": PATIENCE,
}


## 2. Confirm Phase 2 data and record runtime availability

In [2]:
if not DATA_YAML.is_file() or not PHASE2_SUMMARY.is_file():
    raise FileNotFoundError("Run Phase 2 first; the dataset YAML or Phase 2 summary is missing.")
phase2 = json.loads(PHASE2_SUMMARY.read_text(encoding="utf-8"))
if phase2.get("validation_status") != "passed":
    raise ValueError("Phase 2 is not marked passed.")
dataset = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
dataset_root = Path(dataset["path"])
if not dataset_root.is_absolute():
    dataset_root = (PROJECT_ROOT / dataset_root).resolve()
for split in ("train", "val", "test"):
    split_dir = (dataset_root / dataset[split]).resolve()
    if not split_dir.is_dir():
        raise FileNotFoundError(f"{split} directory not found: {split_dir}")
    print(f"{split}: {sum(1 for p in split_dir.iterdir() if p.is_file())} images")
try:
    import torch
    torch_available = True
    print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
except ImportError:
    torch_available = False
    print("PyTorch is not installed in this .venv.")
try:
    import ultralytics
    ultralytics_available = True
    from ultralytics.data.utils import check_det_dataset
    resolved_dataset = check_det_dataset(str(DATA_YAML))
    print("Ultralytics dataset root:", resolved_dataset["path"])
    print("Ultralytics:", ultralytics.__version__)
except ImportError:
    ultralytics_available = False
    print("Ultralytics is not installed in this .venv.")
print("Python:", platform.python_version())
try:
    gpu_probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                               capture_output=True, text=True, timeout=10)
    gpu_names = [line.strip() for line in gpu_probe.stdout.splitlines() if line.strip()]
except (FileNotFoundError, subprocess.TimeoutExpired):
    gpu_names = []
print("Detected GPUs:", gpu_names or "none reported by nvidia-smi")
runtime_record = {
    "checked_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "gpu_names": gpu_names,
    "torch_available": torch_available, "ultralytics_available": ultralytics_available,
    "training_enabled": RUN_TRAINING, "model": MODEL_WEIGHTS,
    "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "batch": BATCH_SIZE, "seed": SEED,
}
(PROJECT_ROOT / "results" / "phase3_runtime.json").write_text(json.dumps(runtime_record, indent=2) + "\n", encoding="utf-8")
print("Training enabled:", RUN_TRAINING)


train: 1257 images
val: 266 images
test: 276 images


PyTorch: 2.15.0+cu132 | CUDA: True
Ultralytics dataset root: ./data/processed
Ultralytics: 8.4.215
Python: 3.14.9
Detected GPUs: ['NVIDIA RTX 4000 Ada Generation']
Training enabled: True


## 3. Train E0 (optional) and save the best checkpoint

In [3]:
best_checkpoint = RUN_PATH / "weights" / "best.pt"
if RUN_TRAINING:
    if not torch_available or not ultralytics_available:
        raise RuntimeError(
            "Install a PyTorch wheel compatible with this CUDA driver, then install ultralytics in .venv. "
            "Do not use a CPU torch wheel if you intend to train on the GPU."
        )
    signature_matches = False
    if best_checkpoint.is_file() and SIGNATURE_PATH.is_file():
        signature_matches = json.loads(SIGNATURE_PATH.read_text(encoding="utf-8")) == training_signature
    if best_checkpoint.is_file() and signature_matches and not FORCE_RETRAIN:
        print("Baseline checkpoint matches the current dataset and settings; preserving it:", best_checkpoint)
    else:
        if RUN_PATH.exists():
            from datetime import datetime
            stale_root = PROJECT_ROOT / "archive" / "stale_runs" / "phase3_baseline"
            stale_root.mkdir(parents=True, exist_ok=True)
            stale_path = stale_root / datetime.now().strftime("%Y%m%dT%H%M%S_%f")
            shutil.move(str(RUN_PATH), str(stale_path))
            print("Archived stale/forced-retrain run:", stale_path)
        from ultralytics import YOLO
        model = YOLO(MODEL_WEIGHTS)
        model.train(
            data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
            patience=PATIENCE, seed=SEED, project=str(RUN_DIR), name="E0_baseline",
            exist_ok=True, device=0 if torch.cuda.is_available() else "cpu",
            workers=4, deterministic=True,
        )
        if best_checkpoint.is_file():
            SIGNATURE_PATH.write_text(json.dumps(training_signature, indent=2) + "\n", encoding="utf-8")
        print("Best checkpoint:", best_checkpoint)
else:
    print("Training skipped. Set RUN_TRAINING = True after installing PyTorch and Ultralytics.")


Ultralytics 8.4.215 🚀 Python-3.14.9 torch-2.15.0+cu132 CUDA:0 (NVIDIA RTX 4000 Ada Generation, 20019MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./configs/neu_det_yolo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E0_baseline, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=12, perspective=0.0, plots=True, pose=12.0

Overriding model.yaml nc=80 with nc=6



                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                


... (1,768 outputs left out)

             scratches         45         78      0.701      0.923      0.906      0.569


Speed: 0.1ms preprocess, 0.9ms inference, 0.0ms loss, 0.5ms postprocess per image


Results saved to ./outputs/checkpoints/phase3_baseline/E0_baseline


Best checkpoint: ./outputs/checkpoints/phase3_baseline/E0_baseline/weights/best.pt


## 4. Evaluate a completed checkpoint on validation only

In [4]:
def save_validation_metrics(checkpoint, experiment_name):
    from ultralytics import YOLO
    model = YOLO(str(checkpoint))
    result = model.val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
                       project=str(PROJECT_ROOT / "outputs" / "logs"), name=experiment_name, exist_ok=True,
                       device=0 if torch.cuda.is_available() else "cpu")
    metrics = getattr(result, "results_dict", {})
    box = result.box
    class_rows = []
    for position, class_id in enumerate(list(getattr(box, "ap_class_index", []))):
        class_id = int(class_id)
        precision = float(box.p[position]) if len(box.p) > position else None
        recall = float(box.r[position]) if len(box.r) > position else None
        class_rows.append({
            "class_id": class_id, "class_name": result.names[class_id],
            "precision": precision, "recall": recall,
            "f1": (2 * precision * recall / (precision + recall)) if precision is not None and recall is not None and precision + recall else 0.0,
            "ap50": float(box.ap50[position]) if len(box.ap50) > position else None,
            "ap50_95": float(box.ap[position].mean()) if len(box.ap) > position else None,
        })
    record = {
        "experiment": experiment_name, "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint), "dataset_fingerprint": dataset_fingerprint,
        "split": "val", "metrics": {str(k): float(v) for k, v in metrics.items()},
        "per_class": class_rows,
        "macro_F1": sum(row["f1"] for row in class_rows) / max(len(class_rows), 1),
        "inference_speed_ms_per_image": getattr(result, "speed", {}),
        "model": MODEL_WEIGHTS, "epochs": EPOCHS, "imgsz": IMAGE_SIZE, "seed": SEED,
    }
    METRICS_PATH.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
    return record

if best_checkpoint.is_file() and torch_available and ultralytics_available:
    validation_record = save_validation_metrics(best_checkpoint, "E0_baseline_validation")
    display(pd.DataFrame([validation_record["metrics"]]))
    display(pd.DataFrame(validation_record["per_class"]))
elif best_checkpoint.is_file():
    print("Checkpoint exists, but PyTorch/Ultralytics is unavailable for validation.")
else:
    print("No baseline checkpoint yet. The test split has not been read.")

Ultralytics 8.4.215 🚀 Python-3.14.9 torch-2.15.0+cu132 CUDA:0 (NVIDIA RTX 4000 Ada Generation, 20019MiB)


Model summary (fused): 73 layers, 3,133,922 parameters, 0 gradients, 8.6 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2020.1±804.9 MB/s, size: 15.2 KB)


val: Scanning ./data/processed/labels/val.cache... 266 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 266/266 185.9Mit/s 0.0s

val: ./data/processed/images/val/crazing_120_3a37b2b9.jpg: 1 duplicate labels removed


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 5% ╸─────────── 1/17 3.5s/it 1.1s<56.5s

... (14 outputs left out)

Speed: 0.3ms preprocess, 1.7ms inference, 0.0ms loss, 0.5ms postprocess per image


Results saved to ./outputs/logs/E0_baseline_validation


,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),fitness
0,0.697163,0.694523,0.757753,0.434756,0.434756


,class_id,class_name,precision,recall,f1,ap50,ap50_95
0,0,crazing,0.628770,0.306515,0.412125,0.465682,0.169670
1,1,inclusion,0.791381,0.809211,0.800197,0.856742,0.448163
2,2,patches,0.780558,0.902256,0.837006,0.916477,0.605886
3,3,pitted_surface,0.784345,0.731343,0.756918,0.833127,0.520939
4,4,rolled-in_scale,0.486210,0.494737,0.490436,0.568346,0.291696
5,5,scratches,0.711714,0.923077,0.803732,0.906142,0.572181


## Phase 3 outputs

Checkpoint: outputs/checkpoints/phase3_baseline/E0_baseline/weights/best.pt  
Validation metrics: results/phase3_baseline_metrics.json  
The test split is reserved for Phase 6.